## Using LettuceDetect with LlamaIndex

Here we build an application that can answer questions about content on a web-page. The demo uses LLMs in a RAG context. In addition to getting the LLM output we also run hallucination detection using LettuceDetect. Using LettuceDetect we can provide a token level confidence abount the output the LLM produced. The answer is highlighed to show parts of the answer that are very likely hallucinated.

This demo uses the OpenAI API and LettuceDetect together with LlamaIndex. The demo is inspired by the LangChain tutorial [Build a Retrieval Augmented Generation (RAG) App](https://python.langchain.com/docs/tutorials/rag/). We build a similar RAG application that also includes hallucination detection using LlamaIndex Workflows.

**Prerequisites**:
  - The `OPENAI_API_KEY` environment variable is set to a valid OpenAI API Key.
  - The [LettuceDetect Web API](../docs/API.md) is running on `localhost:8000`.

**Other demo notebooks**:
  - [Using LettuceDetect with LangChain](./langchain_simple_demo.ipynb)

In [ ]:
%%capture

# Additional dependencies for this notebook.
%pip install llama_index llama-index-readers-web

In [ ]:
from demo_utils import display_output
from llama_index.core import VectorStoreIndex, get_response_synthesizer
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.prompts import RichPromptTemplate
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.workflow import (
    Context,
    Event,
    StartEvent,
    StopEvent,
    Workflow,
    step,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.readers.web import SimpleWebPageReader

from lettucedetect_api.client import LettuceClient


In [ ]:
# Define LettuceDetect web API client. LettuceDetect Web API must be started
# separately (see docs/API.md).
lettuce_client = LettuceClient("http://127.0.0.1:8000")

In [ ]:
# Define LLM and embedding model. Choose the LLM and embedding model you want to
# use. The pre-selected models are the cheapest options OpenAI as to offer at
# the time of writing this notebook and work very well.
llm = OpenAI(model="gpt-5-nano")
embed_model = OpenAIEmbedding(model_name="text-embedding-3-small")

In [ ]:
# System message for the LLM. "Don't use emojis" is included because emojis can
# cause issues when using LettuceDetect. This will be fixed in the future.
prompt_template = RichPromptTemplate(
    '{% chat role="system" %}\n'
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "If you don't know the answer, just say that you don't know. "
    "Don't use emojis."
    "{% endchat %}\n"
    '{% chat role="user" %}\n'
    "Context: {{ context_str }}\n\nQuestion: {{ query_str }}\n"
    "{% endchat %}\n"
)

In [ ]:
# Create the index outside of the workflow. It is then passed to each
# invocation of the workflow. Choose the web-page you want to chat about.
url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
documents = SimpleWebPageReader(html_to_text=True).load_data([url])
splitter = SentenceSplitter(chunk_size=300, chunk_overlap=60)
index = VectorStoreIndex.from_documents(
    documents=documents,
    embed_model=embed_model,
    transformations=[splitter],
)

In [ ]:
# Define LlamaIndex Workflow. It's input are the already created index and the
# user question. Output is the LLM response together with token level confidences.


class AnswerEvent(Event):
    """An event used to pass output from query to halluciantion detection step."""

    question: str
    answer: str
    context: list[str]


class RAGWorkflow(Workflow):
    @step
    async def query(self, ctx: Context, ev: StartEvent) -> AnswerEvent:
        """Retrive context and generate answer with LLM."""
        question = ev.get("question")
        index = ev.get("index")
        retriever = index.as_retriever(similarity_top_k=2)
        synthesizer = get_response_synthesizer(
            llm=llm, response_mode="simple_summarize", text_qa_template=prompt_template
        )
        query_engine = RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=synthesizer,
        )
        result = query_engine.query(question)
        context = [n.node.text for n in result.source_nodes]
        return AnswerEvent(question=question, answer=result.response, context=context)

    @step
    async def detect_hallucination(self, ctx: Context, ev: AnswerEvent) -> StopEvent:
        """Run hallucination detection on generated answer."""
        hallucination_scores = lettuce_client.detect_token(
            contexts=ev.context, question=ev.question, answer=ev.answer
        )
        return StopEvent(
            result={
                "answer": ev.answer,
                "hallucination_scores": hallucination_scores.predictions,
            }
        )

In [ ]:
workflow = RAGWorkflow()

# Invoke the state graph with a user query. Adding "Be creative." to the end of
# is an easy way to make the model hallucinate. Feel free to experiment by
# changing the question to your liking.
result = await workflow.run(index=index, question="What is Task Decomposition? Be creative.")
display_output(result["hallucination_scores"])